# 🦀 Day 4: `Arc<Mutex<T>>` Patterns & Deadlock Prevention
---

In [ ]:
use std::sync::{Arc, Mutex};
use std::thread;

// Bank account simulation
#[derive(Debug)]
struct BankAccount {
    balance: f64,
    name: String,
}

impl BankAccount {
    fn new(name: &str, balance: f64) -> Self {
        BankAccount { balance, name: name.to_string() }
    }
    fn deposit(&mut self, amount: f64) { self.balance += amount; }
    fn withdraw(&mut self, amount: f64) -> Result<(), String> {
        if self.balance >= amount {
            self.balance -= amount; Ok(())
        } else {
            Err(format!("Insufficient funds in {}", self.name))
        }
    }
}

let account = Arc::new(Mutex::new(BankAccount::new("Savings", 1000.0)));
let mut handles = vec![];

for i in 0..5 {
    let acc = Arc::clone(&account);
    handles.push(thread::spawn(move || {
        let mut a = acc.lock().unwrap();
        a.deposit(100.0);
        println!("  Thread {}: deposited $100, balance: ${:.2}", i, a.balance);
    }));
}
for h in handles { h.join().unwrap(); }
println!("Final: ${:.2}", account.lock().unwrap().balance);

In [ ]:
// Deadlock prevention: always lock in the same order!
// BAD: Thread 1 locks A then B, Thread 2 locks B then A → DEADLOCK
// GOOD: Both threads lock A first, then B

// Safe transfer function
fn transfer(
    from: &Mutex<BankAccount>,
    to: &Mutex<BankAccount>,
    amount: f64,
) -> Result<(), String> {
    let mut from_acc = from.lock().unwrap();
    from_acc.withdraw(amount)?;
    let mut to_acc = to.lock().unwrap();
    to_acc.deposit(amount);
    println!("  Transferred ${:.2}: {} → {}", amount, from_acc.name, to_acc.name);
    Ok(())
}

let checking = Mutex::new(BankAccount::new("Checking", 500.0));
let savings = Mutex::new(BankAccount::new("Savings", 1000.0));
transfer(&checking, &savings, 200.0).unwrap();
transfer(&savings, &checking, 100.0).unwrap();
println!("Checking: ${:.2}", checking.lock().unwrap().balance);
println!("Savings: ${:.2}", savings.lock().unwrap().balance);

## 💡 Deadlock Prevention Rules
1. **Lock ordering**: always acquire locks in the same order
2. **Minimize lock scope**: hold locks as briefly as possible
3. **Use `try_lock()`**: returns immediately if lock unavailable
4. **Prefer channels**: message passing avoids shared state entirely

## 🎯 Key Takeaways
1. `Arc<Mutex<T>>` is the go-to for shared mutable state across threads
2. Always lock in a **consistent order** to prevent deadlocks
3. Keep critical sections (lock held) as **short** as possible
4. `try_lock()` returns `Err` instead of blocking
5. Consider channels as an alternative to shared state

---
**Tomorrow:** Rayon for parallel iterators! ⚡